### 📚**RAG Pipeline for Senior Project Document Retrieval**
#### 🔍**Overview**

This notebook presents a Retrieval-Augmented Generation (RAG) pipeline designed for searching and retrieving information from Senior Project documents.

The dataset consists of academic project reports stored in **text-based PDF format**.
The system processes these documents and transforms them into a searchable vector database.

---
#### 📂Dataset Description
- Source: CE Senior Project Reports
- Format: Text-based PDF files
- Characteristics:
    - Structured content (title, author, advisor, year)
    - Semi-structured metadata (keyword, abstract)
    - Multi-page documents

📌Challenge: Extract both **content** and **metadata** accurately from PDFs.

In [1]:
import pdfplumber
import os
import re
from time import perf_counter

#### 1️⃣Text Scanning

PDF documents are scanned to extract raw text content.
- Each page is processed seperately
- Text is extracted using a PDF parsing module
📌Purpose: Convert PDF files into raw textual data for further processing

In [50]:
# set file's path
file_path = r"data\files_for_evaluation\SENIOR-THE-DEVELOPMENT-OF-BLUETOOTH-LOW-ENERGY-IN-CISCO-WLAN.pdf"

extracted_data = []

with pdfplumber.open(file_path)as pdf:

    total_pages = len(pdf.pages)
    print(f"File: {os.path.basename(file_path)}")
    print(f"Total pages: {total_pages}")

    

    for i, page in enumerate(pdf.pages, start=1):
        text = page.extract_text()
        
        extracted_data.append({
            "source": os.path.basename(file_path),
            "page_number":i,
            "content":text
        })
        print(f"\n===== Page {i} =====")
        print(text)

File: SENIOR-THE-DEVELOPMENT-OF-BLUETOOTH-LOW-ENERGY-IN-CISCO-WLAN.pdf
Total pages: 21

===== Page 1 =====
THE DEVELOPMENT OF BLUETOOTH LOW
ENERGY IN CISCO WLAN
KIATTISAK HASATI
PARICHAT TARAM
PARINYADA AIEAMSAMANG
BACHELOR OF ENGINEERING
IN COMPUTER ENGINEERING
MAE FAH LUANG UNIVERSITY
2022
© COPYRIGHT BY MAE FAH LUANG UNIVERSITY

===== Page 2 =====
THE DEVELOPMENT OF BLUETOOTH LOW
ENERGY IN CISCO WLAN
KIATTISAK HASATI
PARICHAT TARAM
PARINYADA AIEAMSAMANG
A SENIOR PROJECT SUBMITTED TO
MAE FAH LUANG UNIVERSITY IN PARTIAL FULFILLMENT OF
THE REQUIREMENTS FOR THE DEGREE OF
BACHELOR OF ENGINEERING
IN COMPUTER ENGINEERING
MAE FAH LUANG UNIVERSITY
2022
© COPYRIGHT BY MAE FAH LUANG UNIVERSITY

===== Page 3 =====
ii
THE DEVELOPMENT OF BLUETOOTH LOW
ENERGY IN CISCO WLAN
KIATTISAK HASATI
PARICHAT TARAM
PARINYADA AIEAMSAMANG
THIS SENIOR PROJECT HAS BEEN APPROVED
TO BE A PARTIAL FULFILLMENT OF THE REQUIREMENTS
FOR THE DEGREE OF BACHELOR OF ENGINEERING
IN COMPUTER ENGINEERING
2022
EXAMINING COMMITT

#### 2️⃣Text Processing

The extracted text is cleaned and normalized.

- Remove unneccessary symbols and noise
- Standardize spacing and formatting
- Prepare text for structured analysis

📌Purpose: Improve data quality before downstream tasks

In [51]:
processed_data = []

for item in extracted_data:
    text = item["content"]

    if text:
        text = text.replace("\n"," ")
        text = " ".join(text.split())
        text = text.strip()

        processed_data.append({
            "source": item["source"],
            "page_number": item["page_number"],
            "content": text
        })

for item in processed_data:
    print(f"\n===== Processed Page {item['page_number']} =====")
    print(item["content"])


===== Processed Page 1 =====
THE DEVELOPMENT OF BLUETOOTH LOW ENERGY IN CISCO WLAN KIATTISAK HASATI PARICHAT TARAM PARINYADA AIEAMSAMANG BACHELOR OF ENGINEERING IN COMPUTER ENGINEERING MAE FAH LUANG UNIVERSITY 2022 © COPYRIGHT BY MAE FAH LUANG UNIVERSITY

===== Processed Page 2 =====
THE DEVELOPMENT OF BLUETOOTH LOW ENERGY IN CISCO WLAN KIATTISAK HASATI PARICHAT TARAM PARINYADA AIEAMSAMANG A SENIOR PROJECT SUBMITTED TO MAE FAH LUANG UNIVERSITY IN PARTIAL FULFILLMENT OF THE REQUIREMENTS FOR THE DEGREE OF BACHELOR OF ENGINEERING IN COMPUTER ENGINEERING MAE FAH LUANG UNIVERSITY 2022 © COPYRIGHT BY MAE FAH LUANG UNIVERSITY

===== Processed Page 3 =====
ii THE DEVELOPMENT OF BLUETOOTH LOW ENERGY IN CISCO WLAN KIATTISAK HASATI PARICHAT TARAM PARINYADA AIEAMSAMANG THIS SENIOR PROJECT HAS BEEN APPROVED TO BE A PARTIAL FULFILLMENT OF THE REQUIREMENTS FOR THE DEGREE OF BACHELOR OF ENGINEERING IN COMPUTER ENGINEERING 2022 EXAMINING COMMITTEE มามะ Crmno ...........................................

#### 3️⃣Metadata Extraction

Important metadata is extracted from the document, especially from the fifth page.

- Project title
- Author(s)
- Advisor
- Keywords
- Year

📌Technique:
- Rule-based extraction using Regular Expressions (Regex)

📌Challenge:
- Metadata may appear in inconsistent formats
- Some fields (e.g., advisor) may appear in different sections

In [52]:
import re

special_meta = {
    "project_title": None,
    "author": None,
    "advisor": None,
    "keywords": None,
    "year": None
}

# searching for year in page 1
if len(processed_data) >= 1:
    text_page1 = processed_data[0]["content"]
    if text_page1:
        year_match = re.search(r"\b(20\d{2}|19\d{2})\b", text_page1)
        if year_match:
            special_meta["year"] = year_match.group(0)

# searching for other metadata in page 5
if len(processed_data) >= 5:
    text = processed_data[4]["content"]

    if text:
        # Title
        title_match = re.search(r"Title\s+(.*?)\s+Author", text)
        if title_match:
            special_meta["project_title"] = title_match.group(1).strip()

        # Author
        author_match = re.search(r"Author\s+(.*?)\s+Degree", text)
        if author_match:
            special_meta["author"] = author_match.group(1).strip()


        # Advisor
        advisor_match = re.search(
            r"Supervisory(?:\s+Committee)?\s+(.*?)\s+Advisor",
            text
        )

        if advisor_match:
            special_meta["advisor"] = advisor_match.group(1).strip()

        # Keywords
        keywords_match = re.search(r"Keywords?:\s+(.*?)(?:\.|$)", text)
        if keywords_match:
            keywords = [k.strip() for k in keywords_match.group(1).split(",")]
            special_meta["keywords"] = keywords
print(processed_data[4]["content"])
print(special_meta)

iv Title The development of Bluetooth low energy in cisco WLAN Author Mr.Kiattisak Hasati Miss Parichat Taram Miss Parinyada Aieamsamang Degree Bachelor of Engineering (Computer Engineering) Supervisory Aj.Mahamah Sebakor Advisor Committee Aj. Dr.Surapol Committ Vorapatratorn Aj. ee Suppakarn Committ Chansareewittaya ee Abstract This senior project proposed the development of a BLE (Bluetooth Low Energy) that is a IoT(Internet of Things) for tracking the durable articles and display the result through website called “MFU AP Mmap”.Use python to develop the website. This senior project will help crew to find durable articles easily. Keyword: Bluetooth Low Energy (BLE), Tracking, Python, Internet of Things (IoT
{'project_title': 'The development of Bluetooth low energy in cisco WLAN', 'author': 'Mr.Kiattisak Hasati Miss Parichat Taram Miss Parinyada Aieamsamang', 'advisor': 'Aj.Mahamah Sebakor', 'keywords': ['Bluetooth Low Energy (BLE)', 'Tracking', 'Python', 'Internet of Things (IoT'], '

#### 4️⃣Chunking

The processed text is divided into smaller segments (chunks).

- Chunking strategies used:
- Fixed-size chunking
- Recursive text splitting
- Chunk size: 500 characters
- Overlap: 100 characters
- Metadata is attached to each chunk

📌 Purpose:

Enable fine-grained retrieval
Preserve contextual continuity between chunks
Improve semantic search performance

📌 Design Consideration:
Using overlap helps retain important context that may span across chunk boundaries, reducing the risk of losing semantic meaning.

In [53]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 500 # max token of embedding model is 512
overlap = 100 # 20% of chunk_size

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=overlap,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = []

for item in processed_data:
    text = item["content"]

    if not text:
        continue

    split_texts = text_splitter.split_text(text)

    for chunk in split_texts:
        chunks.append({
            "content": chunk,
            "metadata": {
                "source": item["source"],
                "page_number": item["page_number"],
                **special_meta
            }
        })
print(f"Total chunks created: {len(chunks)}")
for item in chunks[0]["metadata"]:
    print(f"{item}: {chunks[0]['metadata'][item]}")




Total chunks created: 45
source: SENIOR-THE-DEVELOPMENT-OF-BLUETOOTH-LOW-ENERGY-IN-CISCO-WLAN.pdf
page_number: 1
project_title: The development of Bluetooth low energy in cisco WLAN
author: Mr.Kiattisak Hasati Miss Parichat Taram Miss Parinyada Aieamsamang
advisor: Aj.Mahamah Sebakor
keywords: ['Bluetooth Low Energy (BLE)', 'Tracking', 'Python', 'Internet of Things (IoT']
year: 2022


#### 5️⃣Embedding

Each text chunk is converted into a numerical vector representation using an embedding model.

- Embedding model: intfloat/multilingual-e5-base
- Supports multilingual text (e.g., English and Thai)
- Captures semantic relationships between text segments

📌 Purpose:

- Transform text into vector space for similarity comparison
- Enable semantic search beyond keyword matching

📌 Implementation:
Each chunk is passed into the embedding model to generate dense vector representations.

In [54]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base")

embeddings = []

for chunk in chunks:
    text = "passage: "+ chunk["content"]

    vector = model.encode(text, normalize_embeddings=True)

    embeddings.append({
        "vector": vector,
        "content":chunk["content"],
        "metadata": chunk["metadata"]
    })

print(f"Dimension of embeddings: {len(embeddings[0]['vector'])}")
print(embeddings[0]["vector"])

Dimension of embeddings: 768
[-1.34340040e-02  5.24836741e-02 -8.81501287e-03  1.00738024e-02
  2.57153269e-02 -6.92945793e-02 -4.67865281e-02 -4.11698893e-02
  2.27756351e-02  1.55927632e-02  7.89474323e-03 -1.53677270e-03
  1.27931446e-01  5.91400750e-02 -4.18560281e-02 -6.19140640e-02
 -5.02833677e-03  1.25388103e-03  2.42823791e-02 -3.37084872e-03
  2.08039563e-02 -1.06791817e-02 -6.22853520e-04 -5.70761273e-03
  4.37245145e-02 -8.79447628e-03 -2.41348967e-02  2.88901627e-02
 -1.46738440e-02  3.39544527e-02  2.49027647e-02 -3.43048014e-02
  2.42210496e-02  7.98545312e-03  3.41243185e-02  4.84318957e-02
  1.91352125e-02 -5.79968505e-02 -5.27330209e-03  3.35059268e-03
  3.52994911e-02  2.44641714e-02 -1.78641006e-02 -5.19479215e-02
  1.85199920e-02 -1.68040395e-02  4.38754447e-02  9.83608328e-03
  3.42237623e-03 -1.77420843e-02  6.02767803e-03  2.43600383e-02
  2.38388795e-02 -4.32102894e-03 -2.67413370e-02 -3.19090076e-02
  1.03439586e-02  4.02516546e-03 -2.51105502e-02  3.76833230e

#### 6️⃣Vector Storing

The generated vectors are stored in a vector database along with their metadata.

- Vector database: Qdrant Cloud
- Stores:
    - Embedding vectors
    - Original text content
    - Metadata (title, author, advisor, etc.)

📌 Purpose:

- Enable fast and scalable similarity search
- Support efficient retrieval of relevant document chunks

📌 Advantage:
Using a cloud-based vector database allows scalability, persistence, and real-time access to embeddings.

In [55]:
import uuid
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct

load_dotenv()

QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")

collection_name="pipeline_test"

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

# Prepare points
points = []

for item in embeddings:
    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=item["vector"].tolist(),
            payload={
                "content":item["content"],
                **item["metadata"]
            }
        )
    )

# Upload by batch
batch_size = 100

for i in range(0, len(points),batch_size):
    client.upsert(
        collection_name=collection_name,
        points=points[i:i+batch_size]
    )

print(f"Uploaded {len(points)} vectors to '{collection_name}'")

Uploaded 45 vectors to 'pipeline_test'


#### ⚙️ Retrieval Pipeline

The retrieval process is designed to efficiently locate the most relevant documents using a combination of preprocessing, filtering, semantic search, and reranking.

---

#### 🔁 Pipeline Flow

User Query  
→ Normalize  
→ Extract Filters  
→ Semantic Search + Filtering  
→ Rerank  
→ Final Context for LLM  

---

#### 🎯 Objective
- Improve retrieval accuracy  
- Reduce noise from irrelevant documents  
- Provide high-quality context for answer generation  

---

In [56]:
# =========================
# 📥 INPUT QUERY
# =========================

query = input("📥 Enter your query: ")

print("\n✅ Your Query:")
print(query)



✅ Your Query:
Do you have project about BLE


### 📦 Imports

This section imports all required libraries for the retrieval pipeline.

- `os`, `dotenv` → manage environment variables  
- `qdrant_client` → connect to vector database  
- `sentence_transformers` → reranking model  
- `langchain_huggingface` → embedding model  

📌 Purpose: Prepare all dependencies for the system.

In [57]:
# =========================
# 📦 BASIC IMPORTS
# =========================

import os
import re
import time
from typing import List, Dict, Optional, Tuple

from dotenv import load_dotenv

### ⚙️ Qdrant Connection

This step initializes the connection to the Qdrant vector database.

📌 Purpose:
- Store and retrieve vector embeddings
- Enable fast similarity search

In [58]:
# =========================
# ⚙️ QDRANT SETUP
# =========================

from qdrant_client import QdrantClient

load_dotenv()

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

if not QDRANT_URL:
    raise ValueError("QDRANT_URL is not set")

if not QDRANT_API_KEY:
    raise ValueError("QDRANT_API_KEY is not set")

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=30,
)

print("✅ Qdrant Connected")

✅ Qdrant Connected


### 🧠 Embedding & Reranking Models

This step loads the models used in the pipeline.

- Embedding model → converts text into vectors  
- CrossEncoder → reranks documents based on relevance  

📌 Purpose:
Improve search accuracy using semantic understanding.

In [59]:
# =========================
# 🧠 MODELS
# =========================

from sentence_transformers import CrossEncoder
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "intfloat/multilingual-e5-base"

embed_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("✅ Models Loaded")

✅ Models Loaded


### ⚙️ 4. Configuration

This section defines key parameters used in the retrieval pipeline.

- **COLLECTION_NAME** → Name of the Qdrant collection storing document embeddings  
- **DEFAULT_TOP_K** → Number of documents retrieved in the initial search  
- **DEFAULT_TOP_N** → Number of final documents after reranking  

📌 Purpose:
- Control retrieval depth (Top-K)
- Control final output size (Top-N)
- Make the system configurable and easy to tune

In [60]:
# =========================
# ⚙️ CONFIG
# =========================

COLLECTION_NAME = "embedding_evaluation"

DEFAULT_TOP_K = 20
DEFAULT_TOP_N = 5

### 🔧 1. Query Normalization

This step cleans and standardizes the input query.

📌 Purpose:
- Handle inconsistent input
- Improve retrieval performance

In [61]:
# =========================
# 🔧 NORMALIZE
# =========================

def normalize_query(query: str) -> str:
    print("\n" + "="*40)
    print("🔧 STEP 1: NORMALIZATION")
    print("="*40)

    print("📥 Raw Query:")
    print("   ", query)

    normalized = " ".join(query.strip().split()).lower()

    print("\n🔡 Lowercase + Cleaned:")
    print("   ", normalized)

    replacements = {
        "methology": "methodology",
        "methodolgy": "methodology",
        "petfeeder": "pet feeder",
    }

    for wrong, correct in replacements.items():
        if wrong in normalized:
            print(f"🔁 Fix typo: '{wrong}' → '{correct}'")
            normalized = normalized.replace(wrong, correct)

    print("\n✅ Final Normalized Query:")
    print("   ", normalized)

    return normalized


# 👉 RUN
normalized_query = normalize_query(query)


🔧 STEP 1: NORMALIZATION
📥 Raw Query:
    Do you have project about BLE

🔡 Lowercase + Cleaned:
    do you have project about ble

✅ Final Normalized Query:
    do you have project about ble


### 🧩 2. Query & Filter Extraction

Extract metadata such as year and document type from the query.

📌 Purpose:
Enable metadata-based filtering for more precise search.

In [62]:
# =========================
# 🧩 EXTRACT FILTERS
# =========================

import re

def extract_query_and_filters(user_query: str):
    print("\n" + "="*40)
    print("🧩 STEP 2: EXTRACT FILTERS")
    print("="*40)

    print("🔍 Processing:", user_query)

    filters = {}

    # หา year
    year_match = re.search(r"\b(19\d{2}|20\d{2})\b", user_query)
    if year_match:
        filters["year"] = year_match.group()
        print("📅 Year detected:", filters["year"])

    # clean query
    clean_query = re.sub(r"\b(19\d{2}|20\d{2})\b", "", user_query)
    clean_query = clean_query.strip()

    print("\n🧹 Clean Query:")
    print("   ", clean_query)

    print("📦 Filters:")
    print("   ", filters)

    return clean_query, filters


# 👉 RUN
clean_query, filters = extract_query_and_filters(normalized_query)


🧩 STEP 2: EXTRACT FILTERS
🔍 Processing: do you have project about ble

🧹 Clean Query:
    do you have project about ble
📦 Filters:
    {}


### 🧱 3. Metadata Filter Construction

Convert extracted filters into Qdrant filter format.

📌 Purpose:
Apply constraints during vector search.

In [63]:
# =========================
# 🧱 BUILD QDRANT FILTER
# =========================

from qdrant_client.models import Filter, FieldCondition, MatchValue, MatchAny

def build_qdrant_filter(filters):
    print("\n" + "="*40)
    print("🧱 STEP 3: BUILD FILTER")
    print("="*40)

    print("📦 Input Filters:", filters)

    if not filters:
        print("⚠️ No filters → search all documents")
        return None

    conditions = []

    for key, value in filters.items():
        print(f"➡️ Adding condition: {key} = {value}") 

        if isinstance(value, list):
            conditions.append(FieldCondition(key=key, match=MatchAny(any=value)))
        else:
            conditions.append(FieldCondition(key=key, match=MatchValue(value=value)))

    q_filter = Filter(must=conditions)

    print("\n✅ Final Filter Object:")
    print(q_filter)

    return q_filter


# 👉 RUN
qdrant_filter = build_qdrant_filter(filters)


🧱 STEP 3: BUILD FILTER
📦 Input Filters: {}
⚠️ No filters → search all documents


### 🔍 4. Semantic Search

Convert query into vector and retrieve similar documents.

📌 Purpose:
Find documents based on meaning instead of keywords.

In [64]:
# =========================
# 🔍 SEMANTIC SEARCH
# =========================

def semantic_search_debug(query, top_k, metadata_filters=None):
    print("\n" + "="*40)
    print("🔍 STEP 4: SEMANTIC SEARCH")
    print("="*40)

    print("Query:", query)

    query_vector = embed_model.embed_query(f"query: {query}")
    print("📏 Vector length:", len(query_vector))
    print()
    print("🔢 Query Vector (first 10 dims):", query_vector[:10])
    print()

    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        limit=top_k,
        with_payload=True,
        query_filter=qdrant_filter,
    )

    print("📊 Retrieved:", len(results))

    docs = []
    for i, p in enumerate(results[:5], 1):
        print(f"\n--- Result {i} ---")
        print("Score:", p.score)
        print("Preview:", p.payload.get("content", "")[:100])

    for p in results:
        if p.payload and p.payload.get("content"):

            docs.append({
                "content": p.payload.get("content"),
                "source": p.payload.get("source"),
                "page_number": p.payload.get("page_number"),
                "project_title": p.payload.get("project_title"),
                "author": p.payload.get("author"),
                "advisor": p.payload.get("advisor"),
                "keywords": p.payload.get("keywords"),
                "year": p.payload.get("year"),
                "retrieval_score": p.score
            })

    return docs



# 👉 RUN
docs = semantic_search_debug(clean_query, 20, filters)


🔍 STEP 4: SEMANTIC SEARCH
Query: do you have project about ble
📏 Vector length: 768

🔢 Query Vector (first 10 dims): [0.00131388904992491, 0.05355026200413704, -0.024737132713198662, 0.009164245799183846, 0.03869831934571266, -0.053169794380664825, -0.013513332232832909, -0.06934922188520432, 0.01777004823088646, 0.03620941936969757]

📊 Retrieved: 20

--- Result 1 ---
Score: 0.8407793
Preview: 2
1.5 Plan
Pre-project Project
Table 1.5.1 Working plan
Senior project 1 Senior project 2
Plan/Month

--- Result 2 ---
Score: 0.82584643
Preview: CHAPTER 3 ANALYSIS AND DESIGN
7
3.1 Process
7
3.2 Activity diagram
9
3.3 Web design
10
3.3.1 Homepag

--- Result 3 ---
Score: 0.8258164
Preview: low-power, low-latency communication between devices. BLE is a variant of the
Bluetooth wireless tec

--- Result 4 ---
Score: 0.82440156
Preview: 4
BLE is supported by a wide range of devices, including smartphones,
tablets, and other computing d

--- Result 5 ---
Score: 0.8195962
Preview: v
CONTENTS
Page
SCHO

### 🧠 5. Reranking

Refine results using a CrossEncoder model.

📌 Purpose:
Improve ranking accuracy of retrieved documents.

In [65]:
# =========================
# 🧠 RERANK (CLEAN OUTPUT)
# =========================

def rerank_debug(query: str, docs: List[Dict], top_n: int):

    print("\n" + "="*60)
    print("🧠 STEP 5: RERANK")
    print("="*60)

    if not docs:
        print("❌ No documents to rerank")
        return [], []

    # ใช้ content ไป rerank
    pairs = [[query, d["content"]] for d in docs]

    scores = reranker.predict(pairs)

    ranked = []

    # รวม metadata + rerank score
    for doc, score in zip(docs, scores):

        ranked.append({
            "content": doc["content"],
            "source": doc["source"],
            "page_number": doc["page_number"],
            "project_title": doc["project_title"],
            "author": doc["author"],
            "advisor": doc["advisor"],
            "keywords": doc["keywords"],
            "year": doc["year"],
            "retrieval_score": doc["retrieval_score"],
            "rerank_score": float(score)
        })

    # sort ตาม rerank score
    ranked = sorted(
        ranked,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    top_docs = ranked[:top_n]

    print("\n🏆 FINAL TOP RESULTS\n")

    for i, item in enumerate(top_docs, 1):

        print("-"*60)
        print(f"🥇 Rank {i}")

        print(f"⭐ Rerank Score: {item['rerank_score']:.10f}")
        print(f"📄 Source: {item['source']}")
        print(f"📘 Page: {item['page_number']}")
        print(f"📚 Title: {item['project_title']}")
        print(f"👤 Author: {item['author']}")
        print(f"🎓 Advisor: {item['advisor']}")
        print(f"📅 Year: {item['year']}")
        print(f"🏷️ Keywords: {item['keywords']}")

        print("\n📝 Preview:")
        print(item["content"][:300])
        print()

    return ranked, top_docs


# 👉 RUN
ranked_results, final_results = rerank_debug(
    clean_query,
    docs,
    5
)


🧠 STEP 5: RERANK

🏆 FINAL TOP RESULTS

------------------------------------------------------------
🥇 Rank 1
⭐ Rerank Score: 2.8901345730
📄 Source: SENIOR-THE-DEVELOPMENT-OF-BLUETOOTH-LOW-ENERGY-IN-CISCO-WLAN.pdf
📘 Page: 16
📚 Title: THE DEVELOPMENT OF BLUETOOTH LOW
👤 Author: ENERGY IN CISCO WLAN, KIATTISAK HASATI, PARICHAT TARAM, PARINYADA AIEAMSAMANG
🎓 Advisor: None
📅 Year: 2022
🏷️ Keywords: Bluetooth Low Energy (BLE), Tracking, Python, Internet of

📝 Preview:
8
With the foundational aspects in place, we embarked on an ambitious
endeavor to enhance the project's functionality further. Leveraging Bluetooth
Low Energy (BLE) technology, we developed sophisticated Python code to track
and monitor BLE devices' movements. The code dynamically identified the
dev

------------------------------------------------------------
🥇 Rank 2
⭐ Rerank Score: -0.9847919345
📄 Source: SENIOR-THE-DEVELOPMENT-OF-BLUETOOTH-LOW-ENERGY-IN-CISCO-WLAN.pdf
📘 Page: 5
📚 Title: THE DEVELOPMENT OF BLUETOOTH LOW
👤 Au

In [66]:
# =========================
# 🔢 SCORE DEBUG (TOP-N ONLY)
# =========================

def print_rerank_scores(ranked, top_n=5, show_diff=True):
    print("\n" + "="*60)
    print(f"🔢 STEP 5.1: RERANK SCORES DEBUG (TOP {top_n})")
    print("="*60)

    if not ranked:
        print("❌ No ranked results")
        return

    top_ranked = ranked[:top_n]  # 🔥 เอาแค่ top_n

    print(f"📄 Showing top {len(top_ranked)} docs\n")

    # 🔥 แสดง score
    for i, (_, score) in enumerate(top_ranked, 1):
        print(f"Rank {i:>2} | Score: {score:.10f}")

    # 🔥 diff เฉพาะ top_n
    if show_diff:
        print("\n🔍 Score Differences:")
        for i in range(len(top_ranked)-1):
            diff = top_ranked[i][1] - top_ranked[i+1][1]
            print(f"Rank {i+1} - Rank {i+2}: {diff:.10f}")

In [67]:
# =========================
# 📝 PREVIEW DEBUG AFTER RERANK
# =========================

def print_rerank_preview(query, docs, top_n=5):

    print("\n" + "="*60)
    print("📝 STEP 5.2: RERANK PREVIEW DEBUG")
    print("="*60)

    if not docs:
        print("❌ No documents")
        return

    # ใช้ content สำหรับ rerank
    pairs = [[query, d["content"]] for d in docs]

    scores = reranker.predict(pairs)

    ranked = []

    # รวม metadata + score
    for doc, score in zip(docs, scores):

        ranked.append({
            "content": doc["content"],
            "source": doc["source"],
            "page_number": doc["page_number"],
            "project_title": doc["project_title"],
            "author": doc["author"],
            "advisor": doc["advisor"],
            "keywords": doc["keywords"],
            "year": doc["year"],
            "retrieval_score": doc["retrieval_score"],
            "rerank_score": float(score)
        })

    # sort ตาม rerank score
    ranked = sorted(
        ranked,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    top_docs = ranked[:top_n]

    # print preview
    for i, item in enumerate(top_docs, 1):

        clean = " ".join(item["content"].split())

        print("-"*60)
        print(f"🥇 Rank {i}")
        print(f"⭐ Score: {item['rerank_score']:.10f}")
        print(f"📄 Source: {item['source']}")
        print(f"📘 Page: {item['page_number']}")
        print(f"📚 Title: {item['project_title']}")

        print("\n📝 Preview:")
        print(clean[:300])
        print()

In [68]:
print(print_rerank_preview(clean_query, docs, 5))


📝 STEP 5.2: RERANK PREVIEW DEBUG
------------------------------------------------------------
🥇 Rank 1
⭐ Score: 2.8901345730
📄 Source: SENIOR-THE-DEVELOPMENT-OF-BLUETOOTH-LOW-ENERGY-IN-CISCO-WLAN.pdf
📘 Page: 16
📚 Title: THE DEVELOPMENT OF BLUETOOTH LOW

📝 Preview:
8 With the foundational aspects in place, we embarked on an ambitious endeavor to enhance the project's functionality further. Leveraging Bluetooth Low Energy (BLE) technology, we developed sophisticated Python code to track and monitor BLE devices' movements. The code dynamically identified the dev

------------------------------------------------------------
🥇 Rank 2
⭐ Score: -0.9847919345
📄 Source: SENIOR-THE-DEVELOPMENT-OF-BLUETOOTH-LOW-ENERGY-IN-CISCO-WLAN.pdf
📘 Page: 5
📚 Title: THE DEVELOPMENT OF BLUETOOTH LOW

📝 Preview:
iv Title The development of Bluetooth low energy in cisco WLAN Author Mr.Kiattisak Hasati Miss Parichat Taram Miss Parinyada Aieamsamang Degree Bachelor of Engineering (Computer Engineering) Superviso